# 🏦 AI-Based Loan Approval System

This notebook demonstrates a complete end-to-end Machine Learning classification project to predict loan approvals. It is organized into 20 strict sections.

## 1. Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report, roc_curve, auc
import joblib

import warnings
warnings.filterwarnings('ignore')
sns.set_theme(style="whitegrid")

## 2. Load Dataset

In [ ]:
df = pd.read_csv('../datasets/train_u6lujuX_CVtuZ9i.csv')
df.head()

## 3. Display Dataset Information

In [ ]:
df.info()

## 4. Check Dataset Shape

In [ ]:
print('Dataset Shape:', df.shape)

## 5. Check Data Types

In [ ]:
df.dtypes

## 6. Check Missing Values

In [ ]:
df.isnull().sum()

## 7. Handle Missing Values

In [ ]:
# Categorical: Fill with Mode
for col in ['Gender', 'Married', 'Dependents', 'Self_Employed', 'Loan_Amount_Term', 'Credit_History']:
    df[col].fillna(df[col].mode()[0], inplace=True)
    
# Numerical: Fill with Median
df['LoanAmount'].fillna(df['LoanAmount'].median(), inplace=True)

print("Missing values after handling:")
print(df.isnull().sum())

## 8. Remove Duplicates

In [ ]:
print('Duplicates before:', df.duplicated().sum())
df.drop_duplicates(inplace=True)

## 9. Encode Categorical Variables
We encode variables and save the encoders for later use.

In [ ]:
encoders = {}
cat_cols = ['Gender', 'Married', 'Dependents', 'Education', 'Self_Employed', 'Property_Area', 'Loan_Status']

for col in cat_cols:
    le = LabelEncoder()
    df[col] = le.fit_transform(df[col])
    encoders[col] = le
    
df.head()

## 10. Exploratory Data Analysis
Visualizing relationships and distributions.

In [ ]:
# Histogram & Distribution Plot
plt.figure(figsize=(10,5))
sns.histplot(df['ApplicantIncome'], bins=30, kde=True, color='blue')
plt.title('Applicant Income Distribution')
plt.show()

# Count Plot
plt.figure(figsize=(8,5))
sns.countplot(x='Loan_Status', data=df, palette='Set2')
plt.title('Loan Status Count')
plt.show()

# Box Plot
plt.figure(figsize=(8,5))
sns.boxplot(x='Loan_Status', y='ApplicantIncome', data=df)
plt.title('Applicant Income vs Loan Status')
plt.show()

# Correlation Heatmap
plt.figure(figsize=(12,8))
sns.heatmap(df.corr(), annot=True, cmap='coolwarm', fmt=".2f")
plt.title('Correlation Heatmap')
plt.show()

## 11. Feature Engineering
Dropping columns like Loan_ID that do not provide predictive value.

In [ ]:
if 'Loan_ID' in df.columns:
    df.drop('Loan_ID', axis=1, inplace=True)
print("Columns ready for training:", df.columns.tolist())

## 12. Train Test Split

In [ ]:
X = df.drop('Loan_Status', axis=1)
y = df['Loan_Status']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print("X_train shape:", X_train.shape)
print("X_test shape:", X_test.shape)

## 13. Train Multiple Classification Models

In [ ]:
models = {
    'Logistic Regression': LogisticRegression(max_iter=1000),
    'Decision Tree': DecisionTreeClassifier(random_state=42),
    'Random Forest': RandomForestClassifier(n_estimators=100, random_state=42),
    'SVM': SVC(probability=True, random_state=42)
}

trained_models = {}
for name, model in models.items():
    model.fit(X_train, y_train)
    trained_models[name] = model
    print(f"{name} trained successfully.")

## 14. Compare Model Performance

In [ ]:
performance = {}
for name, model in trained_models.items():
    preds = model.predict(X_test)
    acc = accuracy_score(y_test, preds)
    performance[name] = acc
    print(f"{name} Accuracy: {acc:.4f}")

## 15. Select Best Model

In [ ]:
best_model_name = max(performance, key=performance.get)
print(f"🥇 Best Model Selected: {best_model_name}")
best_model = trained_models[best_model_name]

## 16. Evaluate Model

In [ ]:
y_pred = best_model.predict(X_test)
y_prob = best_model.predict_proba(X_test)[:, 1]

print("Accuracy: ", accuracy_score(y_test, y_pred))
print("Precision: ", precision_score(y_test, y_pred))
print("Recall: ", recall_score(y_test, y_pred))
print("F1 Score: ", f1_score(y_test, y_pred))

print("\nClassification Report:\n", classification_report(y_test, y_pred))

# Confusion Matrix
cm = confusion_matrix(y_test, y_pred)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
plt.title('Confusion Matrix')
plt.show()

# ROC Curve
fpr, tpr, thresholds = roc_curve(y_test, y_prob)
roc_auc = auc(fpr, tpr)
plt.plot(fpr, tpr, color='darkorange', lw=2, label=f'ROC curve (area = {roc_auc:.2f})')
plt.plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('Receiver Operating Characteristic (ROC)')
plt.legend(loc="lower right")
plt.show()

# Feature Importance (if Random Forest or Decision Tree)
if hasattr(best_model, 'feature_importances_'):
    importances = best_model.feature_importances_
    plt.figure(figsize=(10,6))
    sns.barplot(x=importances, y=X.columns)
    plt.title('Feature Importance')
    plt.show()


## 17. Save Final Model using Joblib

In [ ]:
import os
os.makedirs('../models', exist_ok=True)
joblib.dump(best_model, '../models/loan_model.pkl')
print("Model saved to '../models/loan_model.pkl'")

## 18. Save Encoder

In [ ]:
joblib.dump(encoders, '../models/encoder.pkl')
print("Encoders saved to '../models/encoder.pkl'")

## 19. Show Prediction Examples

In [ ]:
sample = X_test.iloc[[0]]
pred = best_model.predict(sample)
prob = best_model.predict_proba(sample)[0]

print("Sample Input:")
print(sample)
print(f"\nPrediction: {'Approved' if pred[0] == 1 else 'Rejected'}")
print(f"Confidence: {prob[pred[0]]*100:.2f}%")

## 20. Conclusion
The notebook successfully demonstrated data ingestion, cleaning, EDA, feature encoding, model selection, evaluation using multiple metrics and charts, and serialization of the artifacts (model and encoder) for deployment via Streamlit.